# 10 Minutes to Dask cuDF

This notebook focuses on **Dask cuDF** for distributed GPU DataFrames. For cuDF-only content, see the [10 Minutes to cuDF](10min.ipynb) notebook.

[Dask cuDF](https://github.com/rapidsai/cudf/tree/main/python/dask_cudf) extends Dask to use cuDF GPU DataFrames as partitions.

Creating a `cudf.Series` and `dask_cudf.Series`.

In [ ]:
import datetime as dt
import os

import cudf
import cupy as cp
import dask_cudf
import pandas as pd

cp.random.seed(12)

# Create base data used in Dask examples
s = cudf.Series([1, 2, 3, None, 4])
df = cudf.DataFrame(
    {
        "a": list(range(20)),
        "b": list(reversed(range(20))),
        "c": list(range(20)),
    }
)
pdf = pd.DataFrame({"a": [0, 1, 2, 3], "b": [0.1, 0.2, None, 0.3]})
df["agg_col1"] = [1 if x % 2 == 0 else 0 for x in range(len(df))]
df["agg_col2"] = [1 if x % 3 == 0 else 0 for x in range(len(df))]
df_a = cudf.DataFrame()
df_a["key"] = ["a", "b", "c", "d", "e"]
df_a["vals_a"] = [float(i + 10) for i in range(5)]
df_b = cudf.DataFrame()
df_b["key"] = ["a", "c", "e"]
df_b["vals_b"] = [float(i + 100) for i in range(3)]

# For Dask examples that need date_df and categorical gdf
date_df = cudf.DataFrame()
date_df["date"] = pd.date_range("11/20/2018", periods=72, freq="D")
date_df["value"] = cp.random.sample(len(date_df))
gdf = cudf.DataFrame(
    {"id": [1, 2, 3, 4, 5, 6], "grade": ["a", "b", "b", "a", "a", "b"]}
)
gdf["grade"] = gdf["grade"].astype("category")

# Create output dir for CSV/parquet examples
if not os.path.exists("example_output"):
    os.mkdir("example_output")
df.to_csv("example_output/foo.csv", index=False)

# For map_partitions and date query examples


def add_ten(num):
    return num + 10


search_date = dt.datetime.strptime("2018-11-23", "%Y-%m-%d")

In [ ]:
ds = dask_cudf.from_cudf(s, npartitions=2)
# Note the call to head here to show the first few entries, unlike
# cuDF objects, Dask-cuDF objects do not have a printing
# representation that shows values since they may not be in local
# memory.
ds.head(n=3)

Creating a `cudf.DataFrame` and a `dask_cudf.DataFrame` by specifying values for each column.

Now we will convert our cuDF dataframe into a Dask-cuDF equivalent. Here we call out a key difference: to inspect the data we must call a method (here `.head()` to look at the first few values). In the general case (see the end of this notebook), the data in `ddf` will be distributed across multiple GPUs.

In this small case, we could call `ddf.compute()` to obtain a cuDF object from the Dask-cuDF object. In general, we should avoid calling `.compute()` on large dataframes, and restrict ourselves to using it when we have some (relatively) small postprocessed result that we wish to inspect. Hence, throughout this notebook we will generally call `.head()` to inspect the first few values of a Dask-cuDF dataframe, occasionally calling out places where we use `.compute()` and why.

*To understand more of the differences between how cuDF and Dask cuDF behave here, visit the [10 Minutes to Dask](https://docs.dask.org/en/stable/10-minutes-to-dask.html) tutorial after this one.*

In [ ]:
ddf = dask_cudf.from_cudf(df, npartitions=2)
ddf.head()

Creating a `cudf.DataFrame` from a pandas `Dataframe` and a `dask_cudf.Dataframe` from a `cudf.Dataframe`.

*Note that best practice for using dask-cuDF is to read data directly into a `dask_cudf.DataFrame` with `read_csv` or other builtin I/O routines (discussed below).*

In [ ]:
dask_gdf = dask_cudf.from_cudf(gdf, npartitions=2)
dask_gdf.head(n=2)

In [ ]:
ddf.head(2)

In [ ]:
ddf.sort_values(by="b").head()

Selecting a single column, which initially yields a `cudf.Series` or `dask_cudf.Series`. Calling `compute` results in a `cudf.Series` (equivalent to `df.a`).

In [ ]:
ddf["a"].head()

In [ ]:
ddf.loc[2:5, ["a", "b"]].head()

Selecting via integers and integer slices, like numpy/pandas. Note that this functionality is not available for Dask-cuDF DataFrames.

In [ ]:
ddf[ddf.b > 15].head(n=3)

Note here we call `compute()` rather than `head()` on the Dask-cuDF dataframe since we are happy that the number of matching rows will be small (and hence it is reasonable to bring the entire result back).

In [ ]:
ddf.query("b == 3").compute()

You can also pass local variables to Dask-cuDF queries, via the `local_dict` keyword. With standard cuDF, you may either use the `local_dict` keyword or directly pass the variable via the `@` keyword. Supported logical operators include `>`, `<`, `>=`, `<=`, `==`, and `!=`.

In [ ]:
dask_cudf_comparator = 3
ddf.query("b == @val", local_dict={"val": dask_cudf_comparator}).compute()

Applying functions to a `Series`. Note that applying user defined functions directly with Dask cuDF is not yet implemented. For now, you can use [map_partitions](http://docs.dask.org/en/stable/generated/dask.dataframe.DataFrame.map_partitions.html) to apply a function to each partition of the distributed dataframe.

In [ ]:
ddf["a"].map_partitions(add_ten).head()

In [ ]:
ddf.a.value_counts().head()

In [ ]:
ds = dask_cudf.from_cudf(s, npartitions=2)
ds.str.lower().head(n=4)

In [ ]:
ds2 = dask_cudf.from_cudf(s, npartitions=2)
dask_cudf.concat([ds2, ds2]).head(n=3)

In [ ]:
ddf_a = dask_cudf.from_cudf(df_a, npartitions=2)
ddf_b = dask_cudf.from_cudf(df_b, npartitions=2)

merged = ddf_a.merge(ddf_b, on=["key"], how="left").head(n=4)
merged

Like [pandas](https://pandas.pydata.org/docs/user_guide/groupby.html), cuDF and Dask-cuDF support the [Split-Apply-Combine groupby paradigm](https://doi.org/10.18637/jss.v040.i01).

In [ ]:
df["agg_col1"] = [1 if x % 2 == 0 else 0 for x in range(len(df))]
df["agg_col2"] = [1 if x % 3 == 0 else 0 for x in range(len(df))]

ddf = dask_cudf.from_cudf(df, npartitions=2)

In [ ]:
ddf.groupby("agg_col1").sum().compute()

In [ ]:
ddf.groupby(["agg_col1", "agg_col2"]).sum().compute()

In [ ]:
ddf.groupby("agg_col1").agg({"a": "max", "b": "mean", "c": "sum"}).compute()

Transposing a dataframe, using either the `transpose` method or `T` property. Currently, all columns must have the same type. Transposing is not currently implemented in Dask cuDF.

In [ ]:
date_ddf = dask_cudf.from_cudf(date_df, npartitions=2)
date_ddf.query(
    "date <= @search_date", local_dict={"search_date": search_date}
).compute()

In [ ]:
dgdf = dask_cudf.from_cudf(gdf, npartitions=2)
dgdf.head(n=3)

Accessing the categories of a column. Note that this is currently not supported in Dask-cuDF.

Converting a cuDF and Dask-cuDF `DataFrame` to a pandas `DataFrame`.

To convert the first few entries to pandas, we similarly call `.head()` on the Dask-cuDF dataframe to obtain a local cuDF dataframe, which we can then convert.

In [ ]:
ddf.head().to_pandas()

In contrast, if we want to convert the entire frame, we need to call `.compute()` on `ddf` to get a local cuDF dataframe, and then call `to_pandas()`, followed by subsequent processing. This workflow is less recommended, since it both puts high memory pressure on a single GPU (the `.compute()` call) and does not take advantage of GPU acceleration for processing (the computation happens on in pandas).

In [ ]:
ddf.compute().to_pandas().head()

Converting a cuDF or Dask-cuDF `DataFrame` to a numpy `ndarray`.

In [ ]:
ddf.compute().to_numpy()

Converting a cuDF or Dask-cuDF `Series` to a numpy `ndarray`.

In [ ]:
ddf["a"].compute().to_numpy()

Converting a cuDF or Dask-cuDF `DataFrame` to a PyArrow `Table`.

In [ ]:
ddf.head().to_arrow()

In [ ]:
ddf.compute().to_csv("example_output/foo_dask.csv", index=False)

Note that for the Dask-cuDF case, we use `dask_cudf.read_csv` in preference to `dask_cudf.from_cudf(cudf.read_csv)` since the former can parallelize across multiple GPUs and handle larger CSV files that would fit in memory on a single GPU.

In [ ]:
ddf = dask_cudf.read_csv("example_output/foo_dask.csv")
ddf.head()

Reading all CSV files in a directory into a single `dask_cudf.DataFrame`, using the star wildcard.

In [ ]:
ddf = dask_cudf.read_csv("example_output/*.csv")
ddf.head()

Writing to parquet files from a `dask_cudf.DataFrame` using cuDF's parquet writer under the hood.

In [ ]:
ddf.to_parquet("example_output/ddf_parquet_files")

## Dask Performance Tips

Like Apache Spark, Dask operations are [lazy](https://en.wikipedia.org/wiki/Lazy_evaluation). Instead of being executed immediately, most operations are added to a task graph and the actual evaluation is delayed until the result is needed.

Sometimes, though, we want to force the execution of operations. Calling `persist` on a Dask collection fully computes it (or actively computes it in the background), persisting the result into memory. When we're using distributed systems, we may want to wait until `persist` is finished before beginning any downstream operations. We can enforce this contract by using `wait`. Wrapping an operation with `wait` will ensure it doesn't begin executing until all necessary upstream operations have finished.

The snippets below provide basic examples, using `LocalCUDACluster` to create one dask-worker per GPU on the local machine. For more detailed information about `persist` and `wait`, please see the Dask documentation for [persist](https://docs.dask.org/en/latest/api.html#dask.persist) and [wait](https://docs.dask.org/en/latest/futures.html#distributed.wait). Wait relies on the concept of Futures, which is beyond the scope of this tutorial. For more information on Futures, see the Dask [Futures](https://docs.dask.org/en/latest/futures.html) documentation. For more information about multi-GPU clusters, please see the [dask-cuda](https://github.com/rapidsai/dask-cuda) library (documentation is in progress).

First, we set up a GPU cluster. With our `client` set up, Dask-cuDF computation will be distributed across the GPUs in the cluster.

In [ ]:
import time

from dask.distributed import Client
from dask_cuda import LocalCUDACluster

cluster = LocalCUDACluster()
client = Client(cluster)

### Persisting Data

Next, we create our Dask-cuDF DataFrame and apply a transformation, storing the result as a new column.

In [ ]:
nrows = 10000000

df2 = cudf.DataFrame({"a": cp.arange(nrows), "b": cp.arange(nrows)})
ddf2 = dask_cudf.from_cudf(df2, npartitions=16)
ddf2["c"] = ddf2["a"] + 5
ddf2

Because Dask is lazy, the computation has not yet occurred. We can see that there are sixty-four tasks in the task graph and we're using about 330 MB of device memory on each GPU. We can force computation by using `persist`. By forcing execution, the result is now explicitly in memory and our task graph only contains one task per partition (the baseline).

### Wait
Depending on our workflow or distributed computing setup, we may want to `wait` until all upstream tasks have finished before proceeding with a specific function. This section shows an example of this behavior, adapted from the Dask documentation.

First, we create a new Dask DataFrame and define a function that we'll map to every partition in the dataframe.

In [ ]:
import random

nrows = 10000000

df1 = cudf.DataFrame({"a": cp.arange(nrows), "b": cp.arange(nrows)})
ddf1 = dask_cudf.from_cudf(df1, npartitions=100)


def func(df):
    time.sleep(random.randint(1, 10))
    return (df + 5) * 3 - 11